# A2 Q2 (part 2) - Two-stage retrieve-then-rank: the "after" numbers

Scores Stage 1 (BM25, frozen-embedding cosine) and Stage 2 (the LightGBM
re-ranker trained in `src/reranker_training_kaggle.ipynb`) on one common
population of impressions, and reports the difference with a **paired**
bootstrap 95% CI.

Inputs, all produced upstream:

| artifact | produced by |
|---|---|
| `reranker_features.parquet` | `src/feature_engineering.ipynb` (A2 Q1) |
| `article_embeddings.parquet` | `src/compute_embeddings_kaggle.ipynb` (A1 Q3) |
| `{bm25,embedding}_topk.parquet` | `src/bm25_retrieval.ipynb`, `src/embedding_retrieval.ipynb` |
| `reranker_model_{dataset}.txt` | `src/reranker_training_kaggle.ipynb` (Kaggle) |
| `eval_metrics.json` | `src/evaluation_harness.ipynb` (A1 Q4) - the full-population reference |

Outputs `data/processed/{dataset}/reranker_eval_{split}.parquet` and
`data/processed/{dataset}/reranker_eval_metrics.json`.

One dataset per kernel via `RERANK_EVAL_DATASETS`, same convention as
`EVAL_DATASETS`/`SCORE_DATASETS`; see `reranker_evaluation.py` for the
one-command wrapper.

## Setup

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil

import lightgbm as lgb
import numpy as np
import polars as pl

from cs4406m26_assignment1c1.evaluation import (
    auc_impression,
    mrr,
    ndcg_at_k,
    bootstrap_ci,
    paired_bootstrap_ci,
)
from cs4406m26_assignment1c1.reranker import (
    FEATURE_COLUMNS,
    KEY_COLUMNS,
    feature_matrix,
    topk_membership_pairs,
    before_after_comparison_table,
)
from cs4406m26_assignment1c1.retrieval import build_stage1_scorers, RECENT_N_CLICKS


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
PROGRESS_LOG = ROOT / "build_progress.log"
CHECKPOINT_DIR = DATA_DIR / "_reranker_eval_checkpoints"
CHUNK_SIZE = 50_000  # impressions per checkpointed chunk, as in reranker_scores.ipynb

# Impressions drawn per (dataset, split) for the head-to-head. Every method is
# scored on this same set, which is what makes paired_bootstrap_ci applicable.
#
# Sampled rather than exhaustive because the re-ranker consumes bm25_score and
# embedding_score as features, so evaluating it on an impression requires a
# full Stage-1 scoring pass over that impression. At the throughput measured
# in build_progress.log (526 impressions/s on ebnerd_large, 325/s on
# mind_large) the complete val+test populations cost ~7.5 hours for
# ebnerd_large's 14,245,374 impressions and ~41 minutes for mind_large's
# 807,988. 200,000 per split costs ~13 and ~20 minutes respectively for both
# splits together.
#
# The sample is not assumed to be representative: write_reranker_eval_metrics
# re-measures the bm25 and embedding baselines on it and compares them against
# the full-population values already in eval_metrics.json. Those baselines were
# computed over every impression, so agreement there is direct evidence that
# the sample is unbiased, and disagreement would invalidate the comparison
# rather than hide inside it.
EVAL_IMPRESSIONS = 200_000
EVAL_SEED = 0

SPLITS = ["val", "test"]
METHODS = ["bm25", "embedding", "reranker"]
BASELINE_METHODS = ["bm25", "embedding"]  # the "before" side of the Q2 comparison
NDCG_K_VALUES = [5, 10]
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 0

BUILD_LARGE_ONLY = True
_DEFAULT_DATASETS = (
    ["ebnerd_large", "mind_large"]
    if BUILD_LARGE_ONLY
    else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"]
)
# Same env-var convention as EVAL_DATASETS/SCORE_DATASETS: one dataset per
# kernel keeps two large BM25 indexes and embedding matrices from being
# resident at once, which is what pinned free RAM at ~0.3GB of 15.7GB during
# A1's WinError 10055 investigation (SPEC.md Q4 #9).
_env_datasets = os.environ.get("RERANK_EVAL_DATASETS")
DATASETS = _env_datasets.split(",") if _env_datasets else _DEFAULT_DATASETS


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] reranker_eval: {message}\n")
        f.flush()


def sink_parquet_atomic(lf: pl.LazyFrame, path: Path) -> None:
    """Write-then-rename, same discipline as every other persisted artifact
    here: a direct sink leaves a truncated file behind if the process is
    killed mid-write, and callers treat existence as completeness."""
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    lf.sink_parquet(tmp_path)
    os.replace(tmp_path, path)


log_progress(f"reranker_evaluation started (datasets={DATASETS})")

store = {}
for name in DATASETS:
    model_path = DATA_DIR / name / f"reranker_model_{name}.txt"
    meta_path = DATA_DIR / name / f"reranker_metadata_{name}.json"
    if not model_path.exists():
        raise FileNotFoundError(
            f"missing {model_path}. Train it with src/reranker_training_kaggle.ipynb on "
            "Kaggle and download the model + metadata into that directory (see README.md)."
        )
    metadata = json.loads(meta_path.read_text(encoding="utf-8"))
    # The booster identifies features positionally, so a metadata column order
    # that disagrees with FEATURE_COLUMNS means feature_matrix would hand it
    # the right numbers in the wrong slots -- silently, with no error and a
    # plausible-looking score distribution. Checked here rather than trusted.
    if metadata["feature_columns"] != FEATURE_COLUMNS:
        raise ValueError(
            f"{name}: model was trained on a different feature order.\n"
            f"  model: {metadata['feature_columns']}\n"
            f"  local: {FEATURE_COLUMNS}"
        )
    store[name] = {
        "articles": pl.read_parquet(DATA_DIR / name / "articles.parquet", columns=["article_id", "title", "abstract"]),
        "history_path": DATA_DIR / name / "history.parquet",
        "embeddings_path": DATA_DIR / name / "article_embeddings.parquet",
        "behaviors_path": DATA_DIR / name / "behaviors.parquet",
        "features_path": DATA_DIR / name / "reranker_features.parquet",
        "booster": lgb.Booster(model_file=str(model_path)),
        "metadata": metadata,
    }
    log_progress(f"  {name}: articles + booster loaded ({metadata['best_iteration']} trees)")

{name: {"articles": store[name]["articles"].height, "trees": store[name]["booster"].num_trees()} for name in DATASETS}

{'mind_large': {'articles': 104151, 'trees': 45}}

In [2]:
def test_setup() -> None:
    for name in DATASETS:
        booster = store[name]["booster"]
        assert booster.num_feature() == len(FEATURE_COLUMNS), (
            f"{name}: booster expects {booster.num_feature()} features, FEATURE_COLUMNS has {len(FEATURE_COLUMNS)}"
        )
        for path_key in ["history_path", "embeddings_path", "behaviors_path", "features_path"]:
            assert store[name][path_key].exists(), f"{name}: missing {store[name][path_key]}"
        # Guards the whole comparison: A1's harness and this notebook must
        # define a user's query over the same history window, or "before" and
        # "after" are not measuring the same system.
        assert RECENT_N_CLICKS == 20, RECENT_N_CLICKS
    assert METHODS[-1] == "reranker" and set(BASELINE_METHODS) < set(METHODS)


test_setup()
print("setup OK:", {name: store[name]["metadata"]["val_auc_per_impression"] for name in DATASETS})

setup OK: {'mind_large': 0.6157967333316782}


## Evaluation population

Every method is scored on the *same* impressions. That is not a convenience:
`paired_bootstrap_ci` resamples one index set and applies it to both methods'
per-impression metric arrays, which cancels the shared per-impression
difficulty and is what makes the interval on the difference meaningful.

In [3]:
def sample_eval_impressions(dataset: str, split: str) -> pl.DataFrame:
    """The evaluation population for one (dataset, split): up to
    EVAL_IMPRESSIONS impressions, seeded, returned sorted by user_id.

    Sorted by impression_id *before* sampling, not after: polars' `unique`
    dedupes by hash and gives no ordering guarantee, so sampling straight
    off it drew a different subset on each run even at a fixed seed (the
    same trap SPEC.md A2 Q2 #4 records for the training sample). The
    returned frame is then re-sorted by user_id so consecutive impressions
    share a user and the BM25 scorer's one-entry cache actually hits --
    worth roughly a 10x reduction in get_scores calls at these inview sizes.
    """
    lf = pl.scan_parquet(store[dataset]["behaviors_path"]).filter(pl.col("split") == split)
    ids = lf.select("impression_id").collect()["impression_id"].sort()
    n = min(EVAL_IMPRESSIONS, ids.len())
    if n < ids.len():
        ids = ids.sample(n=n, seed=EVAL_SEED)
    return (
        lf.filter(pl.col("impression_id").is_in(ids.implode()))
        .select("impression_id", "user_id", "article_ids_inview", "article_ids_clicked")
        .collect()
        .sort(["user_id", "impression_id"])
    )


eval_population = {(name, split): sample_eval_impressions(name, split) for name in DATASETS for split in SPLITS}
log_progress(
    "evaluation population: "
    + str({f"{n}/{s}": eval_population[(n, s)].height for n, s in eval_population})
)
{f"{n}/{s}": eval_population[(n, s)].height for n, s in eval_population}

{'mind_large/val': 200000, 'mind_large/test': 200000}

In [4]:
def test_sampling() -> None:
    for name in DATASETS:
        for split in SPLITS:
            df = eval_population[(name, split)]
            available = (
                pl.scan_parquet(store[name]["behaviors_path"])
                .filter(pl.col("split") == split)
                .select(pl.len())
                .collect()
                .item()
            )
            assert df.height == min(EVAL_IMPRESSIONS, available), (name, split, df.height, available)
            assert df["impression_id"].n_unique() == df.height, f"{name}/{split}: duplicate impressions"
            # Every impression must be scorable and gradeable: an empty inview
            # set has nothing to rank, and auc_impression/ndcg are undefined
            # without at least one click.
            assert df["article_ids_inview"].list.len().min() > 0
            assert df["article_ids_clicked"].list.len().min() > 0
            # Re-drawing must give the identical set, or "same impressions for
            # every method" -- the premise of the paired CI -- does not hold
            # across the separate kernels this notebook runs in.
            again = sample_eval_impressions(name, split)
            assert again["impression_id"].to_list() == df["impression_id"].to_list(), (
                f"{name}/{split}: sampling is not reproducible"
            )
        val_ids = set(eval_population[(name, "val")]["impression_id"].to_list())
        test_ids = set(eval_population[(name, "test")]["impression_id"].to_list())
        assert not (val_ids & test_ids), f"{name}: val and test populations overlap"


test_sampling()
print("sampling OK (reproducible, disjoint splits, every impression rankable)")

sampling OK (reproducible, disjoint splits, every impression rankable)


## Stage-1 scoring

BM25 and embedding scores over the evaluation population. These are both
*baselines* in their own right and *features* the re-ranker consumes, which is
why they come from `cs4406m26_assignment1c1.retrieval` rather than a copy
local to this notebook - see that module's docstring for the drift this
prevents.

In [5]:
_scorer_cache = {"dataset": None, "scorers": None}


def get_scorers(dataset: str) -> dict:
    """Stage-1 scorers for `dataset`, built at most once per kernel.

    Construction costs a BM25 index over the whole corpus plus a normalized
    embedding matrix, so building it per split would double a multi-minute
    setup for nothing. The previous dataset's scorers are dropped *before*
    the next set is built rather than being replaced by assignment, so the
    two are never simultaneously reachable -- assigning over the binding
    would keep the old index alive until the new one finished allocating,
    which is exactly the peak-stacking that SPEC.md A2 Q2 #5 records.
    """
    if _scorer_cache["dataset"] != dataset:
        _scorer_cache["dataset"] = None
        _scorer_cache["scorers"] = None
        log_progress(f"  {dataset}: building stage-1 scorers")
        _scorer_cache["scorers"] = build_stage1_scorers(
            store[dataset]["articles"], store[dataset]["history_path"], store[dataset]["embeddings_path"]
        )
        _scorer_cache["dataset"] = dataset
        log_progress(f"  {dataset}: stage-1 scorers ready ({_scorer_cache['scorers']['n_docs']} docs)")
    return _scorer_cache["scorers"]


def score_stage1(dataset: str, split: str) -> pl.DataFrame:
    """`(impression_id, user_id, article_id, clicked, bm25_score,
    embedding_score, in_bm25_top200, in_embedding_top200)` over the sampled
    evaluation population -- one row per candidate in `article_ids_inview`.

    Chunked and checkpointed per CHUNK_SIZE impressions for the same reason
    every long local loop in this project is: a crash costs at most one
    partial chunk, and a rerun skips what already landed.
    """
    final = CHECKPOINT_DIR / dataset / f"{split}_stage1.parquet"
    if final.exists():
        log_progress(f"  {dataset}/{split}: stage-1 scores loaded from checkpoint")
        return pl.read_parquet(final)

    beh = eval_population[(dataset, split)]
    scorers = get_scorers(dataset)
    chunk_dir = CHECKPOINT_DIR / dataset / f"{split}_stage1_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)
    bounds = list(range(0, beh.height, CHUNK_SIZE)) + [beh.height]
    n_chunks = len(bounds) - 1
    log_progress(f"  {dataset}/{split}: scoring {beh.height} impressions in {n_chunks} chunks")

    for c in range(n_chunks):
        chunk_path = chunk_dir / f"chunk_{c:03d}.parquet"
        if chunk_path.exists():
            continue
        start, end = bounds[c], bounds[c + 1]
        rows = beh.slice(start, end - start)
        # Columnar .to_list() then zip, not iter_rows(named=True): the latter
        # builds a dict per row and never finished at these sizes (SPEC.md
        # A2 Q1 #6 measured 10.6x on the same shape of loop).
        imp_col = rows["impression_id"].to_list()
        uid_col = rows["user_id"].to_list()
        inview_col = rows["article_ids_inview"].to_list()
        clicked_col = rows["article_ids_clicked"].to_list()

        out_imp, out_uid, out_aid, out_lab = [], [], [], []
        out_bm25, out_emb = [], []
        for i in range(len(imp_col)):
            ids = list(inview_col[i])
            clicked_set = set(clicked_col[i])
            bm25_scores = scorers["bm25"](uid_col[i], ids, imp_col[i])
            emb_scores = scorers["embedding"](uid_col[i], ids, imp_col[i])
            out_imp.extend([imp_col[i]] * len(ids))
            out_uid.extend([uid_col[i]] * len(ids))
            out_aid.extend(ids)
            out_lab.extend(aid in clicked_set for aid in ids)
            out_bm25.extend(bm25_scores[aid] for aid in ids)
            out_emb.extend(emb_scores[aid] for aid in ids)

        chunk = pl.DataFrame({
            "impression_id": out_imp,
            "user_id": out_uid,
            "article_id": out_aid,
            "clicked_behaviors": out_lab,
            "bm25_score": out_bm25,
            "embedding_score": out_emb,
        })
        tmp = chunk_path.with_suffix(".parquet.tmp")
        chunk.write_parquet(tmp)
        os.replace(tmp, chunk_path)
        log_progress(f"  {dataset}/{split}: chunk {c + 1}/{n_chunks} checkpointed ({start}-{end}, {chunk.height} rows)")

    # Top-200 membership attached columnar, once, after scoring: as a
    # per-user Python set it was 164,222,200 article-id strings per method
    # (see reranker.topk_membership_pairs).
    #
    # The whole tail stays lazy and lands through sink_parquet. An earlier
    # version read the chunks eagerly and .collect()ed each join, which
    # killed the kernel at ebnerd_large scale immediately after the last
    # chunk was written: the exploded top-200 frame is 164M rows per method,
    # and collecting a join against it materializes far more than the 2.19M
    # scored rows suggest. Streaming bounds the peak regardless of how large
    # the top-K side is -- the same shape reranker_scores.ipynb uses.
    scored = pl.concat([pl.scan_parquet(chunk_dir / f"chunk_{c:03d}.parquet") for c in range(n_chunks)])
    for method, flag in [("bm25", "in_bm25_top200"), ("embedding", "in_embedding_top200")]:
        pairs = topk_membership_pairs(pl.scan_parquet(DATA_DIR / dataset / f"{method}_topk.parquet"), flag)
        scored = scored.join(pairs, on=["user_id", "article_id"], how="left").with_columns(
            pl.col(flag).fill_null(False)
        )

    final.parent.mkdir(parents=True, exist_ok=True)
    sink_parquet_atomic(scored, final)
    shutil.rmtree(chunk_dir)
    df = pl.read_parquet(final)
    log_progress(f"  {dataset}/{split}: stage-1 scores done ({df.height} rows)")
    return df


def stage1_needed(dataset: str, split: str) -> bool:
    """Skip Stage-1 entirely when the re-ranked output for this split already
    exists: that file is a superset of the Stage-1 columns, so on a resume the
    scores would be loaded only to be thrown away. At mind_large's ~40
    candidates per impression that is ~8M rows per split of pure waste."""
    return not (DATA_DIR / dataset / f"reranker_eval_{split}.parquet").exists()


stage1_scores = {
    (name, split): score_stage1(name, split)
    for name in DATASETS
    for split in SPLITS
    if stage1_needed(name, split)
}
{f"{n}/{s}": stage1_scores[(n, s)].height for n, s in stage1_scores} or "all splits already re-ranked"

'all splits already re-ranked'

In [6]:
def test_stage1_scores() -> None:
    for name, split in stage1_scores:
        df = stage1_scores[(name, split)]
        beh = eval_population[(name, split)]
        expected_rows = int(beh["article_ids_inview"].list.len().sum())
        assert df.height == expected_rows, (name, split, df.height, expected_rows)
        assert df["impression_id"].n_unique() == beh.height
        assert df.select(pl.col("bm25_score").is_null().any() | pl.col("embedding_score").is_null().any()).item() is False
        assert df["bm25_score"].min() >= 0.0, "BM25 scores are non-negative by construction"
        assert -1.0001 <= df["embedding_score"].min() and df["embedding_score"].max() <= 1.0001, "cosine out of range"
        # Labels came from behaviors.article_ids_clicked here and from
        # feature_engineering.ipynb's own pass there; they are joined in
        # the next cell and must agree, so a disagreement means one of the
        # two passes mislabelled and every metric below would be wrong.
        #
        # Compared against the *distinct* clicked ids per impression, not
        # the raw list length: EB-NeRD records the same article twice in
        # one impression's article_ids_clicked when a user clicked it
        # twice (3,608 such entries in ebnerd_large val; MIND has none).
        # The per-candidate label is boolean, so a repeat click is one
        # positive row, and comparing against the raw length would fail
        # on correct output.
        distinct_clicks = int(beh["article_ids_clicked"].list.unique().list.len().sum())
        assert df["clicked_behaviors"].sum() == distinct_clicks, (
            f"{name}/{split}: clicked labels do not match behaviors"
        )
        # Every clicked article must be in the inview set it was clicked
        # from -- otherwise a positive exists that no method could ever
        # rank, silently capping every metric below its true value.
        assert int(
            beh.select(
                pl.col("article_ids_clicked").list.set_difference(pl.col("article_ids_inview")).list.len()
            ).to_series().sum()
        ) == 0, f"{name}/{split}: clicked articles absent from article_ids_inview"
        # Membership flags are rare but must not be uniformly absent --
        # an all-False column would mean the join silently missed.
        assert df["in_bm25_top200"].sum() > 0 and df["in_embedding_top200"].sum() > 0

    # The scorers must be reproducible across calls: the BM25 one-entry cache
    # and the shared corpus matrix are mutable state, so a second pass over
    # the same user has to return the same numbers. Skipped when every split
    # resumed from a checkpoint, since no scorer was built in that case.
    if not stage1_scores:
        return
    name = next(iter(stage1_scores))[0]
    scorers = get_scorers(name)
    beh = eval_population[(name, "val")]
    uid, ids = beh["user_id"][0], list(beh["article_ids_inview"][0])
    assert scorers["bm25"](uid, ids) == scorers["bm25"](uid, ids)
    assert scorers["embedding"](uid, ids) == scorers["embedding"](uid, ids)


test_stage1_scores()
print("stage-1 scores OK:", {f"{n}/{s}": stage1_scores[(n, s)].height for n, s in stage1_scores})

# The BM25 index and normalized embedding matrix are the largest objects this
# notebook holds, and nothing after this point scores an impression -- the
# re-ranker consumes the persisted scores. Released before the per-split
# metric frames are built so the two peaks do not stack (SPEC.md A2 Q2 #5).
_scorer_cache["dataset"] = None
_scorer_cache["scorers"] = None
log_progress("  stage-1 scorers released")

stage-1 scores OK: {}


## Re-ranker scoring

Joins the behavioural features to the retrieval scores and runs the booster.
The behavioural half is read back from `reranker_features.parquet` rather than
recomputed, so the values are literally the ones the model was fitted on.

In [7]:
def score_reranker(dataset: str, split: str) -> pl.DataFrame:
    """Stage-1 scores joined to the behavioural features, plus the booster's
    own `reranker_score` -- one row per (impression, candidate).

    The join is the point at which training-time and serving-time features
    have to line up. Both sides are produced by the same code as during
    training: the behavioural half by feature_engineering.ipynb (read back
    here, not recomputed) and the retrieval half by the shared
    build_stage1_scorers, so the only thing that changes between fitting and
    scoring is which impressions are involved.
    """
    final = DATA_DIR / dataset / f"reranker_eval_{split}.parquet"
    if final.exists():
        log_progress(f"  {dataset}/{split}: reranker scores loaded from checkpoint")
        return pl.read_parquet(final)

    s1 = stage1_scores[(dataset, split)]
    joined = (
        pl.scan_parquet(store[dataset]["features_path"])
        .filter(pl.col("split") == split)
        .drop("user_id", "dataset", "split")
        .join(s1.lazy(), on=KEY_COLUMNS, how="inner")
        .collect()
    )
    # An inner join that loses rows means some scored candidate has no
    # behavioural features. That would quietly shrink the evaluation
    # population instead of failing, and the shrunken set would no longer be
    # the same impressions the baselines were measured on.
    if joined.height != s1.height:
        raise ValueError(
            f"{dataset}/{split}: {s1.height - joined.height} of {s1.height} scored candidates "
            "have no row in reranker_features.parquet"
        )
    mismatched = joined.filter(pl.col("clicked").ne_missing(pl.col("clicked_behaviors"))).height
    if mismatched:
        raise ValueError(f"{dataset}/{split}: {mismatched} candidates disagree on the clicked label")

    scores = store[dataset]["booster"].predict(feature_matrix(joined))
    out = joined.drop("clicked_behaviors").with_columns(pl.Series("reranker_score", scores, dtype=pl.Float64))

    tmp = final.with_suffix(".parquet.tmp")
    out.write_parquet(tmp)
    os.replace(tmp, final)
    log_progress(f"  {dataset}/{split}: reranker scores done ({out.height} rows)")
    return out


reranked = {(name, split): score_reranker(name, split) for name in DATASETS for split in SPLITS}
{f"{n}/{s}": reranked[(n, s)].height for n, s in reranked}

{'mind_large/val': 8062455, 'mind_large/test': 7490902}

In [8]:
def test_reranker_scores() -> None:
    for name in DATASETS:
        for split in SPLITS:
            df = reranked[(name, split)]
            if (name, split) in stage1_scores:
                assert df.height == stage1_scores[(name, split)].height
            assert set(FEATURE_COLUMNS) <= set(df.columns)
            s = df["reranker_score"]
            assert s.null_count() == 0 and s.min() >= 0.0 and s.max() <= 1.0, "binary objective -> probability"
            # A wrong feature order raises nothing; it produces a model that
            # scores everything nearly alike, giving AUC ~0.5 everywhere. The
            # property that actually matters is that scores separate
            # candidates *within* an impression, because that is all ranking
            # uses. Checked directly rather than through a distinct-value
            # count, which would depend on how many trees the booster happens
            # to have -- the small-track smoke model has 4 and emits 976
            # distinct values, the Kaggle-trained ones have 45-53.
            per_impression = df.group_by("impression_id").agg(
                pl.len().alias("n_candidates"), pl.col("reranker_score").n_unique().alias("n_scores")
            )
            contested = per_impression.filter(pl.col("n_candidates") > 1)
            flat = contested.filter(pl.col("n_scores") == 1).height
            assert flat / max(contested.height, 1) < 0.01, (
                f"{name}/{split}: {flat}/{contested.height} multi-candidate impressions got one "
                "identical score for every candidate"
            )
            assert s.std() > 1e-6, f"{name}/{split}: reranker_score is constant"

    # Re-predicting a slice through feature_matrix must reproduce the stored
    # scores exactly -- this is the end-to-end check that the persisted column
    # order, the booster, and feature_matrix all still agree.
    name = DATASETS[0]
    df = reranked[(name, "val")].head(20_000)
    again = store[name]["booster"].predict(feature_matrix(df))
    assert np.allclose(again, df["reranker_score"].to_numpy(), rtol=0, atol=0), "prediction is not reproducible"

    # Shuffling the feature order must change the predictions: if it did not,
    # the positional-order guard above would be vacuous.
    shuffled = [FEATURE_COLUMNS[i] for i in [3, 0, 7, 1, 9, 2, 11, 4, 13, 5, 10, 6, 14, 8, 12]]
    assert not np.allclose(store[name]["booster"].predict(feature_matrix(df, shuffled)), df["reranker_score"].to_numpy())


test_reranker_scores()
print("reranker scores OK:", {f"{n}/{s}": reranked[(n, s)].height for n, s in reranked})

# `reranked` contains every Stage-1 column plus the features and the model
# score -- the join kept all rows -- so holding both is holding the same
# ~8M rows per split twice at mind_large's ~40 candidates per impression.
# Released now that the height cross-check above has used it; the on-disk
# checkpoint remains, so a resume loses nothing.
stage1_scores.clear()
log_progress("  stage-1 score frames released (superseded by the re-ranked frames)")

reranker scores OK: {'mind_large/val': 8062455, 'mind_large/test': 7490902}


## Per-impression metrics

`auc_impression`, `mrr` and `ndcg_at_k` unchanged from A1 - they were already
method-agnostic, so the re-ranker is evaluated by exactly the same code that
produced the baselines.

In [9]:
def per_impression_metrics(dataset: str, split: str) -> dict:
    """`{method: {metric: np.ndarray}}`, one array entry per impression, with
    every method's arrays in the *same* impression order.

    That shared order is the whole point: paired_bootstrap_ci resamples one
    index set and applies it to two arrays, which is only meaningful if
    position i refers to the same impression in both. Sorting by
    impression_id gives an order that does not depend on which method is
    being scored.

    Group boundaries come from run lengths over the sorted frame rather than
    a group_by().agg() into list columns: the agg would materialize every
    candidate as a Python object again (up to 8M per split on mind_large),
    which is the conversion this project has repeatedly had to undo. Slicing
    flat numpy arrays costs nothing.
    """
    df = reranked[(dataset, split)].sort("impression_id")
    lengths = df.group_by("impression_id", maintain_order=True).len()["len"].to_numpy()
    # cumsum(dtype=np.int64) is load-bearing, not decoration: polars' len()
    # returns UInt32, numpy's cumsum promotes that to uint64, and there is no
    # common integer type for int64 (the literal [0] below) and uint64 -- so
    # numpy resolves the concatenate to float64, and float64 values cannot be
    # used as slice indices. Pinning the accumulator to int64 keeps the whole
    # expression integral.
    offsets = np.concatenate([[0], np.cumsum(lengths, dtype=np.int64)])
    labels = df["clicked"].to_numpy()
    scores = {method: df[f"{method}_score"].to_numpy() for method in METHODS}

    n = len(lengths)
    out = {
        method: {name: np.empty(n, dtype=np.float64) for name in ["auc", "mrr"] + [f"ndcg{k}" for k in NDCG_K_VALUES]}
        for method in METHODS
    }
    for i in range(n):
        start, end = offsets[i], offsets[i + 1]
        label_slice = labels[start:end]
        for method in METHODS:
            score_slice = scores[method][start:end]
            out[method]["auc"][i] = auc_impression(score_slice, label_slice)
            out[method]["mrr"][i] = mrr(score_slice, label_slice)
            for k in NDCG_K_VALUES:
                out[method][f"ndcg{k}"][i] = ndcg_at_k(score_slice, label_slice, k)
    log_progress(f"  {dataset}/{split}: per-impression metrics computed over {n} impressions")
    return out


impression_metrics = {(name, split): per_impression_metrics(name, split) for name in DATASETS for split in SPLITS}
{
    f"{n}/{s}": {m: round(float(impression_metrics[(n, s)][m]["auc"].mean()), 4) for m in METHODS}
    for n, s in impression_metrics
}

{'mind_large/val': {'bm25': 0.5584, 'embedding': 0.6102, 'reranker': 0.6158},
 'mind_large/test': {'bm25': 0.5569, 'embedding': 0.5944, 'reranker': 0.6099}}

In [10]:
def test_metrics() -> None:
    for name in DATASETS:
        for split in SPLITS:
            per = impression_metrics[(name, split)]
            n = eval_population[(name, split)].height
            for method in METHODS:
                for metric, values in per[method].items():
                    assert len(values) == n, (name, split, method, metric)
                    assert np.isfinite(values).all(), f"{name}/{split}/{method}/{metric} has non-finite entries"
                    assert 0.0 <= values.min() and values.max() <= 1.0, f"{metric} out of [0,1]"
            # nDCG@10 >= nDCG@5, but only where the impression has at most 5
            # clicks. ndcg_at_k normalizes by IDCG@k over min(n_pos, k) terms,
            # so an impression with 6 clicks gets a 6-term IDCG@10 against a
            # 5-term IDCG@5: the denominator grows while the numerator need
            # not, and nDCG@10 can legitimately come out lower (a constructed
            # 6-click case gives 1.000 vs 0.892). ebnerd's val split contains
            # exactly one such impression, which is enough to fail an
            # unconditional version of this check on correct output.
            n_pos = (
                reranked[(name, split)]
                .sort("impression_id")
                .group_by("impression_id", maintain_order=True)
                .agg(pl.col("clicked").sum().alias("p"))["p"]
                .to_numpy()
            )
            eligible = n_pos <= min(NDCG_K_VALUES)
            assert (per["reranker"]["ndcg10"][eligible] >= per["reranker"]["ndcg5"][eligible] - 1e-12).all()

    # Independent recomputation of a handful of impressions straight from the
    # persisted frame, so a bug in the offset arithmetic above cannot pass.
    name, split = DATASETS[0], "val"
    df = reranked[(name, split)].sort("impression_id")
    per = impression_metrics[(name, split)]
    checked = 0
    for i, imp in enumerate(df["impression_id"].unique(maintain_order=True).head(50).to_list()):
        one = df.filter(pl.col("impression_id") == imp)
        labels = one["clicked"].to_numpy()
        for method in METHODS:
            s = one[f"{method}_score"].to_numpy()
            assert abs(auc_impression(s, labels) - per[method]["auc"][i]) < 1e-12, (imp, method)
            assert abs(ndcg_at_k(s, labels, 5) - per[method]["ndcg5"][i]) < 1e-12, (imp, method)
        checked += 1
    assert checked == 50


test_metrics()
print("per-impression metrics OK (50 impressions recomputed independently)")

per-impression metrics OK (50 impressions recomputed independently)


## Extended evaluation: slices and beyond-accuracy (A2 Q5)

Q5 asks for every metric — AUC, MRR, nDCG@5, nDCG@10, diversity, novelty,
coverage — with at least two slices (cold-start vs warm, head vs tail) and a
bootstrap CI on each, for the full two-stage pipeline. All three methods get
the identical treatment on the identical impressions, using A1 Q4's
definitions verbatim, so the re-ranker's slices are directly comparable to
the baselines' in `eval_metrics.json`.

`coverage_top10` is defined over this evaluation sample: the share of the
catalogue that appears in *any* impression's top-10 after ranking. That is a
different quantity from A1's corpus-wide top-200 coverage (which measured the
retrieval lists, not a re-ranking) and is labelled separately for that
reason; what it measures here is whether a method concentrates its top slots
on fewer articles than another on the same candidates.

In [11]:
from collections import Counter

from cs4406m26_assignment1c1.evaluation import (
    coverage,
    intra_list_diversity,
    novelty,
    train_click_counts as train_click_counts_from_lists,
)

HEAD_FRACTION = 0.2   # A1 Q4 #4, unchanged
TOP_LIST_K = 10       # nDCG@10's window, and the list ILD/novelty/coverage are computed over


def build_slice_lookups(dataset: str) -> dict:
    """The slice and beyond-accuracy definitions of A1 Q4, reused verbatim so
    the re-ranker is sliced and scored by the same rules its baselines were.

    - cold-start user: empty `article_id_sequence` in history (A1 Q4 #4).
    - head article: top HEAD_FRACTION of train-clicked articles by click
      count; an impression is "head" if any clicked article is one.
    - novelty basis: -log2 of the add-one-smoothed train popularity. The
      `popularity` column already in the scored frames IS
      `evaluation.train_popularity_lookup` (A2 Q1 #7 factored it there so the
      harness and the feature notebook cannot diverge), verified equal to
      the recomputation to 0.0 on 2,000 articles before this cell was
      written -- so novelty is read from the frame rather than recomputed.
    - category lookup for intra-list diversity.
    """
    history = (pl.scan_parquet(DATA_DIR / dataset / "history.parquet")
               .select("user_id", pl.col("article_id_sequence").list.len().alias("n"))
               .collect())
    coldstart_users = set(history.filter(pl.col("n") == 0)["user_id"].to_list())

    train_clicked = (pl.scan_parquet(DATA_DIR / dataset / "behaviors.parquet")
                     .filter(pl.col("split") == "train")
                     .select("article_ids_clicked").collect()["article_ids_clicked"].to_list())
    click_counts = train_click_counts_from_lists(train_clicked)
    del train_clicked
    ever_clicked = sorted(click_counts.items(), key=lambda kv: -kv[1])
    n_head = max(1, int(len(ever_clicked) * HEAD_FRACTION))
    head_articles = {aid for aid, _ in ever_clicked[:n_head]}

    articles = pl.read_parquet(DATA_DIR / dataset / "articles.parquet", columns=["article_id", "category"])
    category_lookup = dict(zip(articles["article_id"].to_list(), articles["category"].to_list()))

    log_progress(f"  {dataset}: slice lookups built ({len(coldstart_users):,} cold-start users, "
                 f"{n_head:,} head articles of {len(ever_clicked):,} ever clicked)")
    return {"coldstart_users": coldstart_users, "head_articles": head_articles,
            "category_lookup": category_lookup, "n_articles": articles.height,
            "n_ever_clicked": len(ever_clicked)}


slice_lookups = {name: build_slice_lookups(name) for name in DATASETS}
{name: {"coldstart_users": len(v["coldstart_users"]), "head_articles": len(v["head_articles"]),
        "n_articles": v["n_articles"]} for name, v in slice_lookups.items()}

{'mind_large': {'coldstart_users': 14086,
  'head_articles': 2891,
  'n_articles': 104151}}

In [12]:
def test_slice_lookups() -> None:
    for name, lk in slice_lookups.items():
        # Head is a strict minority of ever-clicked articles, by construction.
        assert 0 < len(lk["head_articles"]) < lk["n_ever_clicked"]
        assert len(lk["head_articles"]) == max(1, int(lk["n_ever_clicked"] * HEAD_FRACTION))
        assert len(lk["category_lookup"]) == lk["n_articles"]
        # Cross-check against A1's own eval_metrics.json where it exists: the
        # cold-start slice there is null exactly when the set here is empty.
        q4 = DATA_DIR / name / "eval_metrics.json"
        if q4.exists():
            a1 = json.loads(q4.read_text(encoding="utf-8"))["ranking_metrics"]["bm25"]["val"]["cold_start"]["auc"]
            assert (a1 is None) == (len(lk["coldstart_users"]) == 0), (
                f"{name}: A1 reported cold_start={'null' if a1 is None else 'present'} but this "
                f"definition finds {len(lk['coldstart_users'])} cold-start users"
            )


test_slice_lookups()
print("slice lookups OK:", {n: f"{len(v['coldstart_users'])} cold-start, {len(v['head_articles'])} head"
                            for n, v in slice_lookups.items()})

slice lookups OK: {'mind_large': '14086 cold-start, 2891 head'}


In [13]:
def per_impression_extended(dataset: str, split: str) -> dict:
    """Per impression, in the SAME order as per_impression_metrics (sorted by
    impression_id): slice flags, and per method the top-10 list's ILD and
    novelty. Also the union of every method's top-10 for coverage.

    Top-10 is taken with a stable argsort on the method's score, which is the
    same tie rule evaluate_ranking used for A1's baselines; a different rule
    would change ILD/novelty on tied candidates and make the comparison
    unfair in a way no metric would flag.
    """
    lk = slice_lookups[dataset]
    df = reranked[(dataset, split)].sort("impression_id")
    lengths = df.group_by("impression_id", maintain_order=True).len()["len"].to_numpy()
    offsets = np.concatenate([[0], np.cumsum(lengths, dtype=np.int64)])
    n = len(lengths)

    users = df["user_id"].to_numpy()
    articles = df["article_id"].to_numpy()
    clicked = df["clicked"].to_numpy()
    # -log2(popularity), per candidate row; popularity is per article and
    # identical to A1's novelty basis (see build_slice_lookups).
    nov = -np.log2(df["popularity"].to_numpy().astype(np.float64))
    scores = {m: df[f"{m}_score"].to_numpy() for m in METHODS}

    is_cold = np.empty(n, dtype=bool)
    is_head = np.empty(n, dtype=bool)
    ild = {m: np.empty(n, dtype=np.float64) for m in METHODS}
    novl = {m: np.empty(n, dtype=np.float64) for m in METHODS}
    top_union = {m: set() for m in METHODS}
    cat = lk["category_lookup"]
    head = lk["head_articles"]
    cold = lk["coldstart_users"]

    for i in range(n):
        s, e = offsets[i], offsets[i + 1]
        is_cold[i] = users[s] in cold
        is_head[i] = any(a in head for a in articles[s:e][clicked[s:e]])
        for m in METHODS:
            order = np.argsort(-scores[m][s:e], kind="stable")[:TOP_LIST_K]
            top_ids = articles[s:e][order]
            ild[m][i] = intra_list_diversity(top_ids, cat)
            novl[m][i] = float(nov[s:e][order].mean())
            top_union[m].update(top_ids.tolist())

    cov = {m: len(top_union[m]) / lk["n_articles"] for m in METHODS}
    log_progress(f"  {dataset}/{split}: extended metrics over {n} impressions "
                 f"({int(is_cold.sum())} cold-start, {int(is_head.sum())} head)")
    return {"is_coldstart": is_cold, "is_head": is_head, "ild": ild, "novelty": novl,
            "coverage_top10": cov, "n_top10_union": {m: len(top_union[m]) for m in METHODS}}


extended = {(name, split): per_impression_extended(name, split) for name in DATASETS for split in SPLITS}
{f"{n}/{s}": {"cold": int(v["is_coldstart"].sum()), "head": int(v["is_head"].sum()),
              "coverage_top10": {m: round(c, 4) for m, c in v["coverage_top10"].items()}}
 for (n, s), v in extended.items()}

{'mind_large/val': {'cold': 5246,
  'head': 63008,
  'coverage_top10': {'bm25': 0.0535, 'embedding': 0.0529, 'reranker': 0.0515}},
 'mind_large/test': {'cold': 6016,
  'head': 27707,
  'coverage_top10': {'bm25': 0.0419, 'embedding': 0.0407, 'reranker': 0.0422}}}

In [14]:
def test_extended() -> None:
    for (name, split), ext in extended.items():
        n = eval_population[(name, split)].height
        assert ext["is_coldstart"].shape == (n,) and ext["is_head"].shape == (n,)
        for m in METHODS:
            assert ext["ild"][m].shape == (n,) and ext["novelty"][m].shape == (n,)
            assert 0.0 <= ext["ild"][m].min() and ext["ild"][m].max() <= 1.0
            assert np.isfinite(ext["novelty"][m]).all() and ext["novelty"][m].min() > 0
            assert 0.0 < ext["coverage_top10"][m] <= 1.0
        # Head/tail must be a genuine partition with both sides populated.
        # The head share is SMALL here, and that is correct: "head" is the top
        # 20% of articles by *train* clicks, while val/test are later days on
        # a temporal split, and news decays -- most of what is clicked later
        # did not exist or was not yet popular during train (3.3% on the demo
        # track). A1's ~90% figure was the head articles' share of *train*
        # clicks, a different quantity. An earlier version of this assertion
        # required > 50% and failed on correct output.
        head_share = ext["is_head"].mean()
        assert 0.0 < head_share < 1.0, (name, split, head_share)
        # Cold-start flags must agree with the lookup used to build them.
        assert int(ext["is_coldstart"].sum()) == sum(
            1 for u in reranked[(name, split)].sort("impression_id")
            .group_by("impression_id", maintain_order=True).agg(pl.col("user_id").first())["user_id"].to_list()
            if u in slice_lookups[name]["coldstart_users"])

    # Independent recomputation for a handful of impressions, straight from
    # the frame, so an offset slip cannot pass.
    name, split = DATASETS[0], "val"
    df = reranked[(name, split)].sort("impression_id")
    ext = extended[(name, split)]
    cat = slice_lookups[name]["category_lookup"]
    for i, imp in enumerate(df["impression_id"].unique(maintain_order=True).head(30).to_list()):
        one = df.filter(pl.col("impression_id") == imp)
        for m in METHODS:
            order = np.argsort(-one[f"{m}_score"].to_numpy(), kind="stable")[:TOP_LIST_K]
            top = one["article_id"].to_numpy()[order]
            assert abs(intra_list_diversity(top, cat) - ext["ild"][m][i]) < 1e-12
            expected_nov = float((-np.log2(one["popularity"].to_numpy()[order])).mean())
            assert abs(expected_nov - ext["novelty"][m][i]) < 1e-9


test_extended()
print("extended metrics OK (ranges, head/tail partition, cold-start agreement, 30 impressions recomputed)")

extended metrics OK (ranges, head/tail partition, cold-start agreement, 30 impressions recomputed)


In [15]:
SLICE_DEFINITIONS = {
    "overall": None,
    "cold_start": lambda ext: ext["is_coldstart"],
    "warm": lambda ext: ~ext["is_coldstart"],
    "head": lambda ext: ext["is_head"],
    "tail": lambda ext: ~ext["is_head"],
}
EXTENDED_METRICS = ["auc", "mrr", "ndcg5", "ndcg10", "ild", "novelty"]


def sliced_bootstrap(dataset: str, split: str) -> dict:
    """`{method: {slice: {metric: {point, ci_lo, ci_hi}}}}` -- A1 Q4's
    compute_bootstrap_metrics shape exactly, so eval_metrics.json and this
    file can be read by the same code and quoted in one table.

    An empty slice (EB-NeRD has no cold-start users) yields null rather than
    a degenerate interval, as A1 did.
    """
    per = impression_metrics[(dataset, split)]
    ext = extended[(dataset, split)]
    out = {}
    for m in METHODS:
        arrays = {**{k: per[m][k] for k in ["auc", "mrr", "ndcg5", "ndcg10"]},
                  "ild": ext["ild"][m], "novelty": ext["novelty"][m]}
        out[m] = {}
        for slice_name, mask_fn in SLICE_DEFINITIONS.items():
            mask = None if mask_fn is None else mask_fn(ext)
            n_rows = len(per[m]["auc"]) if mask is None else int(mask.sum())
            entry = {"n": n_rows}
            for metric in EXTENDED_METRICS:
                if n_rows == 0:
                    entry[metric] = None
                    continue
                vals = arrays[metric] if mask is None else arrays[metric][mask]
                point, lo, hi = bootstrap_ci(vals, BOOTSTRAP_ITERATIONS, BOOTSTRAP_SEED, max_chunk_cells=20_000_000)
                entry[metric] = {"point": point, "ci_lo": lo, "ci_hi": hi}
            out[m][slice_name] = entry
    log_progress(f"  {dataset}/{split}: sliced bootstrap CIs computed")
    return out


sliced_metrics = {(name, split): sliced_bootstrap(name, split) for name in DATASETS for split in SPLITS}
for (name, split), sm in sliced_metrics.items():
    print(f"=== {name}/{split}  AUC by slice")
    print(f"  {'slice':11s}" + "".join(f"{m:>12s}" for m in METHODS) + "        n")
    for sl in SLICE_DEFINITIONS:
        cells = []
        for m in METHODS:
            e = sm[m][sl]
            cells.append(f"{e['auc']['point']:12.4f}" if e["auc"] else f"{'null':>12s}")
        print(f"  {sl:11s}" + "".join(cells) + f"  {sm[METHODS[0]][sl]['n']:>7,}")

=== mind_large/val  AUC by slice
  slice              bm25   embedding    reranker        n
  overall          0.5584      0.6102      0.6158  200,000
  cold_start       0.5000      0.5000      0.5243    5,246
  warm             0.5600      0.6131      0.6183  194,754
  head             0.5749      0.6294      0.6742   63,008
  tail             0.5509      0.6013      0.5890  136,992
=== mind_large/test  AUC by slice
  slice              bm25   embedding    reranker        n
  overall          0.5569      0.5944      0.6099  200,000
  cold_start       0.5000      0.5000      0.5109    6,016
  warm             0.5587      0.5973      0.6130  193,984
  head             0.5887      0.5447      0.5885   27,707
  tail             0.5518      0.6023      0.6133  172,293


In [16]:
def test_sliced() -> None:
    for (name, split), sm in sliced_metrics.items():
        n_all = eval_population[(name, split)].height
        for m in METHODS:
            assert sm[m]["overall"]["n"] == n_all
            # Slices partition: cold+warm == head+tail == overall.
            assert sm[m]["cold_start"]["n"] + sm[m]["warm"]["n"] == n_all
            assert sm[m]["head"]["n"] + sm[m]["tail"]["n"] == n_all
            for sl, entry in sm[m].items():
                for metric in EXTENDED_METRICS:
                    e = entry[metric]
                    if entry["n"] == 0:
                        assert e is None, (name, split, m, sl, metric)
                        continue
                    assert e["ci_lo"] <= e["point"] <= e["ci_hi"], (name, split, m, sl, metric)
            # The overall point must equal the plain mean (bootstrap changes
            # the interval, never the estimate).
            assert abs(sm[m]["overall"]["auc"]["point"] - float(impression_metrics[(name, split)][m]["auc"].mean())) < 1e-12
            assert abs(sm[m]["overall"]["ild"]["point"] - float(extended[(name, split)]["ild"][m].mean())) < 1e-12
        # Consistency with A1: where cold_start is null in eval_metrics.json it
        # must be null here too, for the baselines that A1 measured.
        q4 = DATA_DIR / name / "eval_metrics.json"
        if q4.exists():
            a1 = json.loads(q4.read_text(encoding="utf-8"))["ranking_metrics"]
            for m in BASELINE_METHODS:
                a1_null = a1[m][split]["cold_start"]["auc"] is None
                assert a1_null == (sm[m]["cold_start"]["auc"] is None), (name, split, m)


test_sliced()
print("sliced CIs OK (partitions hold, intervals bracket points, A1 null-slices agree)")

sliced CIs OK (partitions hold, intervals bracket points, A1 null-slices agree)


## Paired comparison and `reranker_eval_metrics.json`

Three things are written: each method's own metrics with a standard bootstrap
CI, the paired difference against each baseline, and a check that the sampled
population reproduces the full-population baselines already in
`eval_metrics.json`.

In [17]:
METRIC_NAMES = ["auc", "mrr"] + [f"ndcg{k}" for k in NDCG_K_VALUES]


def sample_agreement(dataset: str, split: str, summary: dict) -> dict:
    """Does the sampled population reproduce the full-population baselines?

    eval_metrics.json holds bm25/embedding measured over *every* val and test
    impression. Those same two methods are re-measured here on the
    EVAL_IMPRESSIONS-sized sample, so if the sample were skewed -- toward
    short inview sets, heavy users, a particular time window -- it would show
    up as the sampled estimate missing the full-population value. The test is
    whether the sample's own 95% CI covers the full-population point
    estimate, which is the correct direction: the sample is the noisy
    measurement, the full population is the reference.
    """
    path = DATA_DIR / dataset / "eval_metrics.json"
    if not path.exists():
        return {"available": False, "reason": f"{path} not found"}
    full = json.loads(path.read_text(encoding="utf-8"))["ranking_metrics"]
    out = {}
    for method in BASELINE_METHODS:
        reference = full.get(method, {}).get(split, {}).get("overall", {})
        per_metric = {}
        for metric in METRIC_NAMES:
            if metric not in reference or reference[metric] is None:
                continue
            full_point = float(reference[metric]["point"])
            sampled = summary[method][split][metric]
            per_metric[metric] = {
                "sampled": sampled["point"],
                "sampled_ci": [sampled["ci_lo"], sampled["ci_hi"]],
                "full_population": full_point,
                "abs_diff": abs(sampled["point"] - full_point),
                "full_within_sample_ci": bool(sampled["ci_lo"] <= full_point <= sampled["ci_hi"]),
            }
        out[method] = per_metric
    out["available"] = True
    return out


def write_reranker_eval_metrics(dataset: str) -> Path:
    summary = {method: {} for method in METHODS}
    for split in SPLITS:
        per = impression_metrics[(dataset, split)]
        for method in METHODS:
            point_ci = {}
            for metric in METRIC_NAMES:
                point, lo, hi = bootstrap_ci(per[method][metric], BOOTSTRAP_ITERATIONS, BOOTSTRAP_SEED)
                point_ci[metric] = {"point": point, "ci_lo": lo, "ci_hi": hi}
            summary[method][split] = point_ci

    paired = {}
    for split in SPLITS:
        per = impression_metrics[(dataset, split)]
        for baseline in BASELINE_METHODS:
            entry = paired.setdefault(f"{baseline}_vs_reranker", {})
            per_metric = {}
            for metric in METRIC_NAMES:
                diff, lo, hi = paired_bootstrap_ci(
                    per[baseline][metric], per["reranker"][metric], BOOTSTRAP_ITERATIONS, BOOTSTRAP_SEED
                )
                per_metric[metric] = {
                    "mean_diff": diff,
                    "ci_lo": lo,
                    "ci_hi": hi,
                    "excludes_zero": bool(lo > 0 or hi < 0),
                }
            entry[split] = per_metric

    payload = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "dataset": dataset,
        "hyperparameters": {
            "eval_impressions": EVAL_IMPRESSIONS,
            "eval_seed": EVAL_SEED,
            "recent_n_clicks": RECENT_N_CLICKS,
            "ndcg_k_values": NDCG_K_VALUES,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
            "bootstrap_seed": BOOTSTRAP_SEED,
        },
        "population": {
            split: {
                "impressions": eval_population[(dataset, split)].height,
                "candidates": reranked[(dataset, split)].height,
                "positive_rate": float(reranked[(dataset, split)]["clicked"].mean()),
            }
            for split in SPLITS
        },
        "model": {
            "best_iteration": store[dataset]["metadata"]["best_iteration"],
            "train_impressions": store[dataset]["metadata"]["train_impressions"],
            "feature_columns": FEATURE_COLUMNS,
        },
        "ranking_metrics": summary,
        "paired_comparison": paired,
        "sample_agreement": {split: sample_agreement(dataset, split, summary) for split in SPLITS},
        # A2 Q5: A1 Q4's slice shape ({method: {slice: {metric: {point, ci_lo, ci_hi}}}})
        # per split, so eval_metrics.json and this file read identically.
        "extended_metrics": {split: sliced_metrics[(dataset, split)] for split in SPLITS},
        "coverage_top10": {split: extended[(dataset, split)]["coverage_top10"] for split in SPLITS},
        "slice_sizes": {split: {sl: sliced_metrics[(dataset, split)][METHODS[0]][sl]["n"]
                                for sl in SLICE_DEFINITIONS} for split in SPLITS},
        "extended_definitions": {
            "cold_start": "user with an empty history article_id_sequence (A1 Q4 #4)",
            "head": f"any clicked article in the top {HEAD_FRACTION:.0%} of train-clicked articles by count (A1 Q4 #4)",
            "ild_novelty_list": f"top-{TOP_LIST_K} by the method's score, stable argsort",
            "novelty_basis": "-log2(add-one train popularity); the popularity column == evaluation.train_popularity_lookup",
            "coverage_top10": "|union of every impression's top-10| / catalogue size, over this evaluation sample; "
                              "NOT A1's corpus-wide top-200 coverage",
        },
        "before_after": {
            f"{baseline}/{split}": before_after_comparison_table(
                {m: summary[baseline][split][m]["point"] for m in METRIC_NAMES},
                {m: summary["reranker"][split][m]["point"] for m in METRIC_NAMES},
                metrics=tuple(METRIC_NAMES),
            )
            for baseline in BASELINE_METHODS
            for split in SPLITS
        },
    }
    path = DATA_DIR / dataset / "reranker_eval_metrics.json"
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(tmp, path)
    log_progress(f"  {dataset}: reranker_eval_metrics.json written")
    return path


eval_metric_paths = {name: write_reranker_eval_metrics(name) for name in DATASETS}
for name in DATASETS:
    payload = json.loads(eval_metric_paths[name].read_text(encoding="utf-8"))
    print(f"=== {name}")
    for split in SPLITS:
        cells = "  ".join(f"{m:9s} {payload['ranking_metrics'][m][split]['auc']['point']:.4f}" for m in METHODS)
        print(f"  {split:5s} AUC  {cells}")
    for key, per_split in payload["paired_comparison"].items():
        for split in SPLITS:
            d = per_split[split]["auc"]
            flag = "yes" if d["excludes_zero"] else "NO"
            print(f"  {key} ({split}): {d['mean_diff']:+.4f} [{d['ci_lo']:+.4f}, {d['ci_hi']:+.4f}] significant={flag}")

=== mind_large
  val   AUC  bm25      0.5584  embedding 0.6102  reranker  0.6158
  test  AUC  bm25      0.5569  embedding 0.5944  reranker  0.6099
  bm25_vs_reranker (val): +0.0574 [+0.0559, +0.0590] significant=yes
  bm25_vs_reranker (test): +0.0530 [+0.0514, +0.0546] significant=yes
  embedding_vs_reranker (val): +0.0057 [+0.0046, +0.0068] significant=yes
  embedding_vs_reranker (test): +0.0155 [+0.0145, +0.0166] significant=yes


In [18]:
def test_eval_metrics_roundtrip() -> None:
    for name in DATASETS:
        payload = json.loads(eval_metric_paths[name].read_text(encoding="utf-8"))
        assert payload["schema_version"] == 1 and payload["dataset"] == name
        assert payload["model"]["feature_columns"] == FEATURE_COLUMNS
        for method in METHODS:
            for split in SPLITS:
                for metric in METRIC_NAMES:
                    entry = payload["ranking_metrics"][method][split][metric]
                    assert entry["ci_lo"] <= entry["point"] <= entry["ci_hi"], (name, method, split, metric)
                    recomputed = float(impression_metrics[(name, split)][method][metric].mean())
                    assert abs(entry["point"] - recomputed) < 1e-12

        # A paired difference of means must equal the difference of the two
        # point estimates exactly -- the bootstrap only affects the interval,
        # never the estimate, so a mismatch means the arrays were misaligned.
        for baseline in BASELINE_METHODS:
            for split in SPLITS:
                for metric in METRIC_NAMES:
                    d = payload["paired_comparison"][f"{baseline}_vs_reranker"][split][metric]
                    expected = (
                        payload["ranking_metrics"]["reranker"][split][metric]["point"]
                        - payload["ranking_metrics"][baseline][split][metric]["point"]
                    )
                    assert abs(d["mean_diff"] - expected) < 1e-12, (name, baseline, split, metric)
                    assert d["ci_lo"] <= d["mean_diff"] <= d["ci_hi"]
                    assert d["excludes_zero"] == (d["ci_lo"] > 0 or d["ci_hi"] < 0)

        # The sample must reproduce the full-population baselines, or the
        # comparison is measuring a different population than A1 reported.
        for split in SPLITS:
            agreement = payload["sample_agreement"][split]
            if not agreement.get("available"):
                continue
            for method in BASELINE_METHODS:
                for metric, entry in agreement[method].items():
                    assert entry["full_within_sample_ci"], (
                        f"{name}/{split}/{method}/{metric}: sampled {entry['sampled']:.4f} "
                        f"{entry['sampled_ci']} does not cover full-population {entry['full_population']:.4f} "
                        "-- the evaluation sample is not representative"
                    )


def test_extended_roundtrip() -> None:
    for name in DATASETS:
        payload = json.loads(eval_metric_paths[name].read_text(encoding="utf-8"))
        for split in SPLITS:
            ext = payload["extended_metrics"][split]
            for m in METHODS:
                assert set(ext[m]) == set(SLICE_DEFINITIONS)
                assert abs(ext[m]["overall"]["ild"]["point"] - float(extended[(name, split)]["ild"][m].mean())) < 1e-12
                assert 0 < payload["coverage_top10"][split][m] <= 1
            sizes = payload["slice_sizes"][split]
            assert sizes["cold_start"] + sizes["warm"] == sizes["overall"] == sizes["head"] + sizes["tail"]


test_eval_metrics_roundtrip()
test_extended_roundtrip()
print("eval metrics OK (round-trip, paired-difference identity, sample representativeness)")

eval metrics OK (round-trip, paired-difference identity, sample representativeness)


# Manual Verification Complete